# Shrinkage encoding & hierarchical group/allocation blend

Two refinements to your best-performing feature (`ALLOC_TARGET_ENCODING`):

1. **Shrinkage**: right now a 65-observation allocation and a 2400-observation allocation get equally trusted raw sign-rates. Shrinking low-count allocations toward the global mean should reduce noise without losing real signal.
2. **Hierarchical blend**: GROUP has its own real (if smaller, and less stable for GROUP 4) signal. Rather than using ALLOCATION and GROUP as two flat, separate features, blend them hierarchically -- an allocation's estimate shrinks toward its GROUP's rate first, and the GROUP's rate shrinks toward the global rate, so low-confidence allocations fall back on the best available higher-level information rather than jumping straight to the global average.

Both are tested the same way as every feature tonight: standalone correlation/consistency check first, then a proper leakage-safe CV ablation against your current best (52.45-52.47% combined model).

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgbm
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

X_train = pd.read_csv('../data/X_train.csv', index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv', index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv', index_col='ROW_ID')
total_data = pd.concat([X_train, y_train], axis=1)

## 1. Shrinkage estimator -- define and inspect

`shrunk_rate = (raw_rate * n + global_rate * k) / (n + k)`

`k` controls how much weight the global average gets, in units of "pseudo-observations". A small `k` (e.g. 10) barely changes high-count allocations but meaningfully pulls in low-count ones; a large `k` (e.g. 200) shrinks everything toward the global mean more aggressively. Try a few values of `k` and look at the effect on your known extremes (min 38.47%, max 65.89% from earlier) before picking one to test in CV.

In [3]:
def compute_shrunk_alloc_rate(train_targets, train_allocs, k):
    df = pd.DataFrame({'ALLOCATION': train_allocs, 'target': train_targets.values})
    grouped = df.groupby('ALLOCATION')['target']
    raw_rate = grouped.apply(lambda s: (np.sign(s) > 0).mean())
    count = grouped.size()
    global_rate = (np.sign(train_targets) > 0).mean()

    shrunk = (raw_rate * count + global_rate * k) / (count + k)
    return shrunk, raw_rate, count, global_rate

for k in [10, 50, 100, 200]:
    shrunk, raw_rate, count, global_rate = compute_shrunk_alloc_rate(total_data['target'], total_data['ALLOCATION'], k)
    comparison = pd.DataFrame({'raw_rate': raw_rate, 'shrunk_rate': shrunk, 'count': count}).sort_values('count')
    print(f"\n--- k={k} ---")
    print(f"Global rate: {global_rate:.4f}")
    print("Lowest-count allocations (most affected by shrinkage):")
    print(comparison.head(5))
    print(f"Shrunk rate spread: std={shrunk.std():.4f} (raw was {raw_rate.std():.4f})")


--- k=10 ---
Global rate: 0.5072
Lowest-count allocations (most affected by shrinkage):
               raw_rate  shrunk_rate  count
ALLOCATION                                 
ALLOCATION_46  0.421053     0.450753     19
ALLOCATION_14  0.421053     0.450753     19
ALLOCATION_01  0.534591     0.534401   1431
ALLOCATION_92  0.482879     0.483048   1431
ALLOCATION_91  0.485674     0.485824   1431
Shrunk rate spread: std=0.0267 (raw was 0.0274)

--- k=50 ---
Global rate: 0.5072
Lowest-count allocations (most affected by shrinkage):
               raw_rate  shrunk_rate  count
ALLOCATION                                 
ALLOCATION_46  0.421053     0.483467     19
ALLOCATION_14  0.421053     0.483467     19
ALLOCATION_01  0.534591     0.533666   1431
ALLOCATION_92  0.482879     0.483700   1431
ALLOCATION_91  0.485674     0.486401   1431
Shrunk rate spread: std=0.0258 (raw was 0.0274)

--- k=100 ---
Global rate: 0.5072
Lowest-count allocations (most affected by shrinkage):
               raw_r

Pick the `k` that meaningfully pulls in the low-count allocations (check the ~65-count ones specifically) without flattening the overall spread too much -- if `shrunk_rate std` drops close to 0, you've over-shrunk and thrown away the real signal you're trying to keep.

## 2. Hierarchical GROUP -> ALLOCATION blend

Two-level shrinkage: GROUP rate shrinks toward global; ALLOCATION rate shrinks toward its own GROUP's (already-shrunk) rate, not straight to the global average. This lets a low-count allocation fall back on "what's typical for allocations like this one" rather than jumping all the way to the population mean.

In [4]:
def compute_hierarchical_encoding(train_targets, train_allocs, train_groups, k_group=50, k_alloc=50):
    df = pd.DataFrame({
        'ALLOCATION': train_allocs.values,
        'GROUP': train_groups.values,
        'target': train_targets.values,
    })

    global_rate = (np.sign(df['target']) > 0).mean()

    # level 1: GROUP rate shrunk toward global rate
    grp = df.groupby('GROUP')['target']
    group_raw = grp.apply(lambda s: (np.sign(s) > 0).mean())
    group_count = grp.size()
    group_shrunk = (group_raw * group_count + global_rate * k_group) / (group_count + k_group)

    # map each row's group-shrunk rate in as its prior
    df['GROUP_PRIOR'] = df['GROUP'].map(group_shrunk)

    # level 2: ALLOCATION rate shrunk toward ITS OWN group's shrunk rate (not global)
    alloc_grp = df.groupby('ALLOCATION')
    alloc_raw = alloc_grp['target'].apply(lambda s: (np.sign(s) > 0).mean())
    alloc_count = alloc_grp.size()
    # each allocation's own group prior (should be constant within an allocation, take first)
    alloc_group_prior = alloc_grp['GROUP_PRIOR'].first()

    alloc_shrunk = (alloc_raw * alloc_count + alloc_group_prior * k_alloc) / (alloc_count + k_alloc)

    return alloc_shrunk, group_shrunk, global_rate

alloc_hier_rate, group_rate, global_rate = compute_hierarchical_encoding(
    total_data['target'], total_data['ALLOCATION'], total_data['GROUP']
)
print(f"Global rate: {global_rate:.4f}")
print("\nGroup-level shrunk rates:")
print(group_rate)
print(f"\nAllocation-level hierarchical rate spread: std={alloc_hier_rate.std():.4f}")
print(alloc_hier_rate.describe())

Global rate: 0.5072

Group-level shrunk rates:
GROUP
1    0.500522
2    0.509782
3    0.509918
4    0.506006
Name: target, dtype: float64

Allocation-level hierarchical rate spread: std=0.0258
count    278.000000
mean       0.506376
std        0.025834
min        0.387827
25%        0.492927
50%        0.505042
75%        0.516023
max        0.655884
dtype: float64


## 3. CV ablation: flat vs shrunk vs hierarchical encoding

Same combined feature set + LightGBM setup as your best model tonight, swapping only the encoding method. Runs all three encodings in the same loop for a direct, fold-matched comparison.

In [5]:
RET_features = [f'RET_{i}' for i in range(1, 21)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]
TURNOVER_features = ['MEDIAN_DAILY_TURNOVER']

X_train['CROSS_SECTIONAL_AVG_RET1'] = X_train.groupby('TS')['RET_1'].transform('mean')

for i in [3, 5, 10, 15, 20]:
    X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
    X_train[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')

for i in [20]:
    X_train[f'STD_PERF_{i}'] = X_train[RET_features[:i]].std(1)
    X_train[f'ALLOCATIONS_STD_PERF_{i}'] = X_train.groupby('TS')[f'STD_PERF_{i}'].transform('mean')

SIGNED_VOLUME_features_clean = [f'SIGNED_VOLUME_{i}' for i in range(2, 21)]
for i in [3, 5, 10, 15, 20]:
    cols = SIGNED_VOLUME_features_clean[:i - 1]
    X_train[f'AVERAGE_VOLUME_{i}'] = X_train[cols].mean(1)
    X_train[f'ALLOCATIONS_AVERAGE_VOLUME_{i}'] = X_train.groupby('TS')[f'AVERAGE_VOLUME_{i}'].transform('mean')

for i in [20]:
    cols = SIGNED_VOLUME_features_clean[:i - 1]
    X_train[f'STD_VOLUME_{i}'] = X_train[cols].std(1)
    X_train[f'ALLOCATIONS_STD_VOLUME_{i}'] = X_train.groupby('TS')[f'STD_VOLUME_{i}'].transform('mean')

base_features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
base_features += [f'AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]]
base_features += [f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]]
base_features += [f'STD_PERF_{i}' for i in [20]]
base_features += [f'ALLOCATIONS_STD_PERF_{i}' for i in [20]]
base_features += [f'AVERAGE_VOLUME_{i}' for i in [3, 5, 10, 15, 20]]
base_features += [f'ALLOCATIONS_AVERAGE_VOLUME_{i}' for i in [3, 5, 10, 15, 20]]
base_features += [f'STD_VOLUME_{i}' for i in [20]]
base_features += [f'ALLOCATIONS_STD_VOLUME_{i}' for i in [20]]
base_features += ['CROSS_SECTIONAL_AVG_RET1']

X_train_minimal = X_train.copy()

lgbm_params = {
    "objective": "mse", "metric": "mse", "num_threads": 50, "seed": 42,
    "verbosity": -1, 'learning_rate': 1e-2, 'max_depth': 3,
}
NUM_BOOST_ROUND = 500
N_SPLITS = 8
SHRINK_K = 50       # chosen from section 1's inspection -- adjust based on what you saw
K_GROUP = 50
K_ALLOC = 50

train_dates = X_train_minimal['TS'].unique()

encoding_variants = ['flat', 'shrunk', 'hierarchical']
scores = {v: [] for v in encoding_variants}

splits = KFold(n_splits=N_SPLITS, random_state=0, shuffle=True).split(train_dates)

for i, (local_train_dates_ids, local_test_dates_ids) in enumerate(splits):
    local_train_dates = train_dates[local_train_dates_ids]
    local_test_dates = train_dates[local_test_dates_ids]

    local_train_ids = X_train_minimal['TS'].isin(local_train_dates)
    local_test_ids = X_train_minimal['TS'].isin(local_test_dates)

    fold_train_targets = y_train.loc[local_train_ids, 'target']
    fold_train_allocs = X_train_minimal.loc[local_train_ids, 'ALLOCATION']
    fold_train_groups = X_train_minimal.loc[local_train_ids, 'GROUP']

    # --- FLAT (your existing baseline encoding) ---
    flat_df = pd.DataFrame({'ALLOCATION': fold_train_allocs, 'target': fold_train_targets.values})
    flat_rate = flat_df.groupby('ALLOCATION')['target'].apply(lambda s: (np.sign(s) > 0).mean())
    global_rate = (np.sign(fold_train_targets) > 0).mean()

    # --- SHRUNK ---
    shrunk_rate, _, _, _ = compute_shrunk_alloc_rate(fold_train_targets, fold_train_allocs, SHRINK_K)

    # --- HIERARCHICAL ---
    hier_rate, _, _ = compute_hierarchical_encoding(fold_train_targets, fold_train_allocs, fold_train_groups, K_GROUP, K_ALLOC)

    variant_maps = {
        'flat': (flat_rate, global_rate),
        'shrunk': (shrunk_rate, global_rate),
        'hierarchical': (hier_rate, global_rate),
    }

    for variant, (rate_map, fallback) in variant_maps.items():
        X_train_minimal.loc[local_train_ids, 'ALLOC_TARGET_ENCODING'] = (
            X_train_minimal.loc[local_train_ids, 'ALLOCATION'].map(rate_map).fillna(fallback)
        )
        X_train_minimal.loc[local_test_ids, 'ALLOC_TARGET_ENCODING'] = (
            X_train_minimal.loc[local_test_ids, 'ALLOCATION'].map(rate_map).fillna(fallback)
        )

        minimal_features = base_features + ['ALLOC_TARGET_ENCODING']

        X_local_train = X_train_minimal.loc[local_train_ids, minimal_features]
        y_local_train = y_train.loc[local_train_ids, 'target']
        X_local_test = X_train_minimal.loc[local_test_ids, minimal_features]
        y_local_test = y_train.loc[local_test_ids, 'target']

        train_data = lgbm.Dataset(X_local_train, label=y_local_train.values)
        model = lgbm.train(lgbm_params, train_data, num_boost_round=NUM_BOOST_ROUND)
        y_local_pred = model.predict(X_local_test, num_threads=lgbm_params['num_threads'])

        score = accuracy_score((y_local_test > 0).astype(int), (y_local_pred > 0).astype(int))
        scores[variant].append(score)

    print(f"Fold {i+1} - flat: {scores['flat'][-1]*100:.2f}%  shrunk: {scores['shrunk'][-1]*100:.2f}%  hierarchical: {scores['hierarchical'][-1]*100:.2f}%")

print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
for variant in encoding_variants:
    s = scores[variant]
    print(f"{variant}: {np.mean(s)*100:.2f}% (+- {np.std(s)*100:.2f})")

/tmp/ipykernel_690412/2478572225.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_690412/2478572225.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_690412/2478572225.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_690412/2478572225.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_690412/2478572225.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(1)
/tmp/

Fold 1 - flat: 52.34%  shrunk: 52.30%  hierarchical: 52.33%
Fold 2 - flat: 52.61%  shrunk: 52.64%  hierarchical: 52.62%
Fold 3 - flat: 52.34%  shrunk: 52.31%  hierarchical: 52.33%
Fold 4 - flat: 52.50%  shrunk: 52.44%  hierarchical: 52.45%
Fold 5 - flat: 52.00%  shrunk: 52.03%  hierarchical: 52.04%
Fold 6 - flat: 52.93%  shrunk: 52.93%  hierarchical: 52.87%
Fold 7 - flat: 52.54%  shrunk: 52.54%  hierarchical: 52.53%
Fold 8 - flat: 52.49%  shrunk: 52.47%  hierarchical: 52.46%

SUMMARY
flat: 52.47% (+- 0.25)
shrunk: 52.46% (+- 0.25)
hierarchical: 52.45% (+- 0.23)


## 4. Interpreting the result

- If shrunk/hierarchical beat flat on MEAN accuracy: real improvement, likely from cleaning up noisy low-count allocations.
- If they match flat's mean but have TIGHTER std: still a genuine win -- more consistent, more trustworthy predictions, which matters given tonight's finding about public/private leaderboard divergence favouring consistency over variance.
- If they're all within noise of each other: given only 2 allocations had <100 observations, there may simply not have been much noise to clean up in the first place -- the flat encoding was already fairly reliable given how much history most allocations have.
- Try re-running section 3 with different SHRINK_K / K_GROUP / K_ALLOC values if the first pass is inconclusive -- the right amount of shrinkage isn't knowable in advance and is worth a quick manual sweep (e.g. K in [10, 50, 100, 200]).